# 10 · 命令行 CLI

把 Python 脚本从「只能在编辑器里跑」变成「能在终端机接受参数、可重复运行的命令行工具 command-line tool」。

## 学习目标
- 理解程序如何透过命令行参数 command-line arguments 接收外部输入，不需修改原代码即可改变行为。
- 能用 sys.argv 读取简易参数，并说明它的限制。
- 能用 argparse 设计正式 CLI，包含位置参数、选用旗标 flag、类型转换、默认值与自动产生的说明 help。
- 理解 if __name__ == "__main__" 惯例的意义，正确区分「被运行」与「被导入」。
- 认识 subprocess 调用外部程序 external program 的基本概念与使用时机。
- 能把一支脚本设计成接口清楚、可重复运行、可被他人使用的工具。

## 为什么需要命令行接口

命令行接口 command-line interface（CLI）是让程序在终端机 terminal 用文字指令启动，并透过命令行参数 command-line arguments 接收外部输入的方式。

为什么要学：若把参数写死 hard-code 在代码里，每换一组输入就得改原代码、存盘、重跑，无法给别人重复使用。CLI 让「同一支程序」靠外部输入改变行为。

在终端机运行脚本的形状（不运行，仅示意）：

```
python 脚本.py 参数1 参数2 --旗标 值
```

In [ ]:
# 概念：写死参数 vs 从外部接收参数的差异对比

# 写死名字：要换人就得改这一行原代码
def greet_hardcoded():
    name = "Alice"                       # 名字被写死，无法从外部更换
    return f"Hello, {name}!"

# 从外部接收名字：同一支函数可服务任何输入
def greet(name):
    return f"Hello, {name}!"             # 名字由调用端决定，不需改原代码

print(greet_hardcoded())
# 仿真命令行传入不同名字（之后会用 sys.argv / argparse 真正从终端机取得）
for external_name in ["Alice", "Bob", "Cathy"]:
    print(greet(external_name))

## sys.argv 简易参数读取

sys.argv 是一个串列 list，存放启动脚本时在终端机输入的所有文字。它是最底层、最直接的取得参数方式，适合理解原理。

重点提醒：
- 第 0 个元素 `sys.argv[0]` 是脚本名称，真正的参数从索引 1 开始。
- 所有参数都是字符串 string，数字要自己手动做类型转换 type conversion。
- 没有自动验证，参数缺漏或格式错误要自己检查，这也凸显为何需要更正式的工具。

In [ ]:
# 概念：仿真 sys.argv 读取两个数字并相加
import sys

# 注意：在 notebook 里 sys.argv 是 kernel 启动参数，不是我们要的。
#       这里自造一个假的 argv 来示范终端机运行 `python add.py 3 5` 的情况。
fake_argv = ["add.py", "3", "5"]          # [0] 是脚本名，[1:] 才是参数

# 参数数量检查：少于 3 个代表用户没给满两个数字
if len(fake_argv) < 3:
    print("用法：python add.py 数字1 数字2")
else:
    a = float(fake_argv[1])               # 字符串转数字，手动类型转换
    b = float(fake_argv[2])               # 若这里传入非数字会丢出 ValueError
    print(f"{a} + {b} = {a + b}")

# 技巧：实际脚本中改用 sys.argv 取代 fake_argv 即可
print("目前 kernel 的 sys.argv[0]：", sys.argv[0])

## argparse 正式 CLI 设计

argparse 是 Python 标准函数库的 CLI 模块，取代手动处理 sys.argv，自动完成参数解析、错误提示与说明文档，是 Python 标准的 CLI 作法。

核心对象与步骤：
1. 创建 ArgumentParser 对象。
2. 用 add_argument 方法声明位置参数 positional argument。
3. 用 parse_args 解析参数；argparse 会自动产生 help 与用法 usage 消息。

形状（不运行，仅示意）：

```
parser = argparse.ArgumentParser()
parser.add_argument("名称", help="说明")
args = parser.parse_args()
```

In [ ]:
# 概念：用 argparse 改写加法脚本（两个位置参数）
import argparse

# 技巧：在 notebook 里可用 parse_args([...]) 传入仿真参数，
#       实际脚本则直接调用 parse_args()，由终端机提供。
parser = argparse.ArgumentParser(description="把两个数字相加")
parser.add_argument("a", type=float, help="第一个加数")   # 位置参数，顺序固定
parser.add_argument("b", type=float, help="第二个加数")   # type=float 自动转型

args = parser.parse_args(["3", "5"])      # 仿真终端机输入 `python add.py 3 5`
print(f"{args.a} + {args.b} = {args.a + args.b}")

# 注意：argparse 会自动产生说明；下面印出 usage 行示意 -h 的效果
print(parser.format_usage().strip())

## argparse 进阶：旗标、类型与默认值

选用参数 optional argument 以 `--名称` 形式出现（即旗标 flag），可有可无；透过旗标、类型与默认值，让同一支工具支持多种使用模式。

常用设置对照：

| 设置 | 作用 |
|---|---|
| `type=int` | 把输入字符串转成指定类型 |
| `default=值` | 未提供时的默认值 default |
| `action="store_true"` | 布尔开关，出现即为 True |
| `choices=[...]` | 限定只能从清单中选 |
| `required=True` | 设为必填的选用参数 |

In [ ]:
# 概念：文字处理小工具（大小写旗标 + 带默认值的重复次数）
import argparse

parser = argparse.ArgumentParser(description="文字处理小工具")
parser.add_argument("text", help="要处理的文字")              # 位置参数
parser.add_argument("--upper", action="store_true",          # 布尔开关旗标
                    help="转成大写")
parser.add_argument("--repeat", type=int, default=1,         # 带默认值的选用参数
                    help="重复次数（预设 1）")

# 仿真 `python tool.py hello --upper --repeat 3`
args = parser.parse_args(["hello", "--upper", "--repeat", "3"])

result = args.text.upper() if args.upper else args.text       # 旗标决定是否转大写
print((result + " ") * args.repeat)                           # 用预设 / 指定次数重复

# 注意：不给 --repeat 时会自动用 default=1，不会报错
args2 = parser.parse_args(["world"])
print(args2.text, "重复", args2.repeat, "次")

## __main__ 惯例与脚本入口

`if __name__ == "__main__"` 是一个惯例，用来区分文件是「被直接运行」还是「被当作模块 module 导入 import」。

为什么要学：每个文件都有内置变量 `__name__`。被直接运行时它等于 `"__main__"`；被别的文件导入时它等于模块名称。把主流程放进这个判断里，文件就能既当可运行脚本、又能被导入而不误触发主流程，这是可重用代码的基础。

形状（不运行，仅示意）：

```
def main():
    ...
if __name__ == "__main__":
    main()
```

In [ ]:
# 概念：__name__ 在「运行」与「导入」时的值不同

# 在 notebook 中直接运行的保存格，__name__ 就是 "__main__"
print("目前 __name__ =", __name__)

def main():
    print("main() 被调用：这是主流程，只有被直接运行才会跑")

# 注意：被导入时 __name__ 会变成模块名（例如 'mytool'），
#       下面这行就不会触发，避免 import 时误跑主流程。
if __name__ == "__main__":
    main()                                # entry point：程序入口

## subprocess 调用外部程序

subprocess 模块让 Python 程序启动并协调其他外部程序 external program（即创建一个子进程 subprocess）。常用 `subprocess.run`，参数以串列传入。

重点概念：
- 回传码 return code 为 0 通常代表成功，非 0 代表失败。
- 可截取输出 output（标准输出文字）做后续判断。
- 何时用：需要调用系统指令或既有命令行工具，而非用 Python 重写一遍时。
- 安全提醒：参数用串列传入（不要把整串文字交给 shell），避免注入风险。

In [ ]:
# 概念：用 subprocess.run 调用外部程序并判断是否成功
import sys
import subprocess

# 参数以串列传入：这里调用目前的 Python 解释器印出一行字（跨平台都存在）
completed = subprocess.run(
    [sys.executable, "-c", "print('hello from subprocess')"],
    capture_output=True,                  # 截取输出而非直接显示
    text=True,                            # 以文字（str）而非字节处理输出
)

print("回传码 return code：", completed.returncode)   # 0 表示成功
print("截取到的输出：", completed.stdout.strip())

# 技巧：用回传码判断成败，再决定后续流程
if completed.returncode == 0:
    print("外部程序运行成功")
else:
    print("外部程序运行失败")

## 把脚本做成可重复运行的工具

工具化 tool 设计思维：集成前面所学，让他人不需读原代码就能使用一支 CLI。

一支好用工具的接口检查清单：

| 元素 | 设计原则 |
|---|---|
| 位置参数 | 放最关键、必要的输入 |
| 旗标 flag | 放可选的模式切换与调整 |
| 默认值 default | 给合理预设，常见情境免设置即可用 |
| 说明 help | 每个参数都写清楚用途 |
| 结束状态码 exit code | 0 成功、非 0 失败，方便被其他流程组合 composable |

In [ ]:
# 概念：用 argparse + main + exit code 组成一支自足的小工具
import argparse

def build_parser():
    parser = argparse.ArgumentParser(description="判断数值是否超过上限的小工具")
    parser.add_argument("value", type=float, help="要检查的数值")
    parser.add_argument("--limit", type=float, default=100.0,   # 合理默认值
                        help="上限（预设 100）")
    return parser

def main(argv):
    args = build_parser().parse_args(argv)
    over = args.value > args.limit
    print(f"value={args.value} limit={args.limit} -> {'超标' if over else '合格'}")
    # 用结束码回报成败：超标回 1，合格回 0（这里用 return 仿真，脚本中用 sys.exit）
    return 1 if over else 0

# 仿真两种终端机调用，观察结束码差异
print("exit code：", main(["120"]))            # 用预设 limit=100，超标
print("exit code：", main(["80", "--limit", "90"]))  # 合格

## 练习

以下三题由浅入深，皆为复合型但简短。数据自己造，不引用任何外部文件。

In [ ]:
# TODO 1 ·（简单）容积率快查工具（集成：sys.argv 简易参数读取 + 手动类型转换）
#   情境：用命令行输入一栋建筑的楼地板面积 GFA（gross floor area）与占地面积
#         footprint area，立刻算出容积率 FAR（floor area ratio）。
#   要求：
#     1. 用 sys.argv 读取两个参数（GFA、footprint）；它们都是字符串 string，
#        要用 float 手动做类型转换。（notebook 中可先用 fake_argv 仿真）
#     2. 计算 FAR = GFA / footprint，并用 print 印出结果（保留两位小数）。
#   验收：等同在终端机运行 `python far.py 12000 3000`，应看到 `FAR = 4.00`。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）建蔽率检核 CLI（集成：argparse 位置参数 + 旗标 + 类型转换 + 默认值 + __main__ 惯例）
#   情境：审查一笔基地数据，输入占地面积与建筑投影面积，计算建蔽率
#         coverage（building coverage ratio），并依政策上限判定是否合格。
#   要求：
#     1. 用 argparse 设计两个位置参数（site area 基地面积、footprint 投影面积），
#        都用 type=float，并为每个参数写 help。
#     2. 加旗标 --limit（type=float, default=0.6）代表建蔽率上限；
#        再加布尔开关 --verbose（store_true）控制是否印出详细计算过程。
#     3. coverage = footprint / site area；超过 limit 印「超标」、否则印「合格」。
#        把主流程包进 main 函数，并用 if __name__ == "__main__" 调用它
#        （notebook 中可用 parse_args([...]) 仿真输入）。
#   验收：等同运行 `python coverage.py 1000 700 --verbose`，
#         应看到 coverage 约 0.70 且判定为「超标」。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）街廓楼高聚合分析器（集成：argparse 旗标与 choices + numpy 聚合 + 情境前后比较 + exit code）
#   情境：一个街廓 block 内多栋建筑的楼高 height（公尺）数据，做聚合统计，
#         并比较套用「高度乘数」政策情境前后的差异。
#   要求：
#     1. 在程序内用 numpy 自造一组仿真楼高数据，例如
#        heights = np.array([12.0, 30.0, 8.0, 45.0, 21.0, 60.0])（不可引用外部文件）。
#     2. 用 argparse 加 --stat 旗标搭配 choices（限定 mean/max/min）决定聚合统计量；
#        再加 --multiplier（type=float, default=1.0）仿真政策放宽后的高度乘数。
#     3. 对「原始楼高」与「乘上 multiplier 后的楼高」各算一次选定统计量，
#        印出两者与差值；若放宽后的统计量超过 50 公尺，用 sys.exit(1) 回报，
#        否则以 0 结束，让此工具可被其他流程判断成败。
#   验收：等同运行 `python block.py --stat max --multiplier 1.2`，
#         应看到原始 max、放宽后 max 与差值，且因放宽后超过 50 公尺而以 exit code 1 结束。
# TODO: 在这里完成

## 小结

- 命令行参数 command-line arguments 让同一支程序靠外部输入改变行为，不必修改原代码。
- sys.argv 是最底层的取得方式：`[0]` 是脚本名、参数皆为字符串、需手动类型转换与检查。
- argparse 是标准作法：位置参数、选用旗标 flag、type 转型、default 默认值、choices 限定、store_true 开关，并自动产生 help 与 usage。
- `if __name__ == "__main__"` 区分「被运行」与「被导入」，把主流程放进 main 函数是可重用代码的基础。
- subprocess 用串列传入参数调用外部程序，并以回传码 return code 判断成败。
- 好用工具的特质：清楚的接口与 help、合理的默认值、用结束状态码 exit code 表达成败，方便被其他流程组合 composable。